# 02_Practice: 토크나이저 직접 비교하기

## 개요
이 노트북에서는 **BPE**, **WordPiece**, **SentencePiece** 3가지 토크나이저를 실제로 로드하고, 동일한 텍스트에 어떻게 다르게 작동하는지 직접 비교합니다.

### 학습 목표
1. 각 토크나이저 로드 및 기본 사용법 이해
2. 같은 텍스트가 다르게 토크나이징되는 과정 관찰
3. 토큰 개수 비교를 통한 효율성 분석
4. 실제 모델에서의 영향 이해

---

## 1. 환경 설정 및 토크나이저 로드

In [ ]:
# 필요한 라이브러리 임포트
from transformers import AutoTokenizer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (선택사항)
plt.rcParams['font.family'] = 'DejaVu Sans'

print("라이브러리 로드 완료")

### 1.1 3가지 토크나이저 로드

In [ ]:
print("=" * 70)
print("토크나이저 로드")
print("=" * 70)

# 1. GPT-2 (BPE)
print("\n[1] GPT-2 (BPE) 로드 중...")
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(f"✓ GPT-2")
print(f"   - 어휘 크기: {len(gpt2_tokenizer):,}개")
print(f"   - 타입: Byte Pair Encoding")
print(f"   - 특징: 빈도 기준, 공백 처리 (</w>)")

# 2. mBERT (WordPiece)
print("\n[2] mBERT (WordPiece) 로드 중...")
mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
print(f"✓ mBERT")
print(f"   - 어휘 크기: {len(mbert_tokenizer):,}개")
print(f"   - 타입: WordPiece")
print(f"   - 특징: 정보이득 기준, ## 표시")

# 3. XLM-RoBERTa (SentencePiece)
print("\n[3] XLM-RoBERTa (SentencePiece) 로드 중...")
xlm_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
print(f"✓ XLM-RoBERTa")
print(f"   - 어휘 크기: {len(xlm_tokenizer):,}개")
print(f"   - 타입: SentencePiece")
print(f"   - 특징: 다국어 처리, 공백 명시 (▁)")

print("\n" + "=" * 70)
print("모든 토크나이저 로드 완료!")
print("=" * 70)

---

## 2. 테스트 텍스트 정의

In [ ]:
# 테스트 케이스 정의
test_cases = {
    "English": "I love machine learning and natural language processing.",
    "Korean": "기계 학습과 자연어 처리를 좋아합니다.",
    "Mixed": "I love 자연어 processing! Machine learning rocks!"
}

print("\n" + "=" * 70)
print("테스트 케이스")
print("=" * 70)

for lang, text in test_cases.items():
    print(f"\n[{lang}]")
    print(f"텍스트: {text}")

---

## 3. BPE (GPT-2) 토크나이징

In [ ]:
print("\n" + "=" * 70)
print("GPT-2 (BPE) 토크나이징 결과")
print("=" * 70)

gpt2_results = {}

for lang, text in test_cases.items():
    tokens = gpt2_tokenizer.tokenize(text)
    token_ids = gpt2_tokenizer.encode(text)
    
    gpt2_results[lang] = {
        'tokens': tokens,
        'count': len(token_ids),
        'ids': token_ids
    }
    
    print(f"\n테스트 [{lang}] - {len(token_ids)}개 토큰")
    print(f"텍스트: {text}")
    print(f"토큰: {tokens[:20]}{'...' if len(tokens) > 20 else ''}")

---

## 4. WordPiece (mBERT) 토크나이징

In [ ]:
print("\n" + "=" * 70)
print("mBERT (WordPiece) 토크나이징 결과")
print("=" * 70)

mbert_results = {}

for lang, text in test_cases.items():
    tokens = mbert_tokenizer.tokenize(text)
    token_ids = mbert_tokenizer.encode(text)
    # [CLS], [SEP] 제거 (BERT의 특수 토큰)
    actual_count = len(token_ids) - 2
    
    mbert_results[lang] = {
        'tokens': tokens,
        'count': actual_count,
        'ids': token_ids
    }
    
    print(f"\n테스트 [{lang}] - {actual_count}개 토큰 (실제 토큰, [CLS], [SEP] 제외)")
    print(f"텍스트: {text}")
    print(f"토큰: {tokens[:20]}{'...' if len(tokens) > 20 else ''}")

---

## 5. SentencePiece (XLM-RoBERTa) 토크나이징

In [ ]:
print("\n" + "=" * 70)
print("XLM-RoBERTa (SentencePiece) 토크나이징 결과")
print("=" * 70)

xlm_results = {}

for lang, text in test_cases.items():
    tokens = xlm_tokenizer.tokenize(text)
    token_ids = xlm_tokenizer.encode(text)
    # <s>, </s> 제거
    actual_count = len(token_ids) - 2
    
    xlm_results[lang] = {
        'tokens': tokens,
        'count': actual_count,
        'ids': token_ids
    }
    
    print(f"\n테스트 [{lang}] - {actual_count}개 토큰 (실제 토큰, <s>, </s> 제외)")
    print(f"텍스트: {text}")
    print(f"토큰: {tokens[:20]}{'...' if len(tokens) > 20 else ''}")

---

## 6. 결과 비교 분석

In [ ]:
# 비교 데이터프레임 생성
comparison_data = []

for lang in test_cases.keys():
    comparison_data.append({
        'Language': lang,
        'GPT-2': gpt2_results[lang]['count'],
        'mBERT': mbert_results[lang]['count'],
        'XLM': xlm_results[lang]['count']
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "=" * 70)
print("토큰 개수 비교")
print("=" * 70)
print(comparison_df.to_string(index=False))

### 6.1 차이점 분석

In [ ]:
print("\n" + "=" * 70)
print("토크나이저 간 차이점 분석")
print("=" * 70)

for idx, row in comparison_df.iterrows():
    lang = row['Language']
    gpt2 = row['GPT-2']
    mbert = row['mBERT']
    xlm = row['XLM']
    
    print(f"\n[{lang}]")
    print(f"GPT-2 vs mBERT: {gpt2 - mbert:+d}개 ({(gpt2/mbert - 1)*100:+.1f}%)")
    print(f"GPT-2 vs XLM:   {gpt2 - xlm:+d}개 ({(gpt2/xlm - 1)*100:+.1f}%)")
    print(f"mBERT vs XLM:   {mbert - xlm:+d}개 ({(mbert/xlm - 1)*100:+.1f}%)")
    
    # 가장 효율적인 토크나이저
    min_tokens = min(gpt2, mbert, xlm)
    if min_tokens == gpt2:
        best = "GPT-2"
    elif min_tokens == mbert:
        best = "mBERT"
    else:
        best = "XLM"
    print(f"\n→ 가장 효율적: {best} (최소 {min_tokens}개 토큰)")

### 6.2 비용 영향 계산

In [ ]:
# API 비용 예시 (OpenAI GPT-4 기준: 1000토큰당 $0.03)
cost_per_token = 0.00003  # $0.03 per 1000 tokens

print("\n" + "=" * 70)
print("API 비용 영향 (1000토큰당 $0.03 기준)")
print("=" * 70)

cost_data = []

for idx, row in comparison_df.iterrows():
    lang = row['Language']
    gpt2_cost = row['GPT-2'] * cost_per_token
    mbert_cost = row['mBERT'] * cost_per_token
    xlm_cost = row['XLM'] * cost_per_token
    
    cost_data.append({
        'Language': lang,
        'GPT-2 ($)': f"${gpt2_cost:.6f}",
        'mBERT ($)': f"${mbert_cost:.6f}",
        'XLM ($)': f"${xlm_cost:.6f}"
    })
    
    print(f"\n[{lang}]")
    print(f"GPT-2:  ${gpt2_cost:.6f}")
    print(f"mBERT:  ${mbert_cost:.6f} ({(1 - mbert_cost/gpt2_cost)*100:.1f}% 절감)")
    print(f"XLM:    ${xlm_cost:.6f} ({(1 - xlm_cost/gpt2_cost)*100:.1f}% 절감)")

print("\n★ 주의: 장기 사용 시 토크나이저 선택이 비용에 큰 영향!")

---

## 7. 시각화

In [ ]:
# 막대 그래프
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(comparison_df))
width = 0.25

bars1 = ax.bar(x - width, comparison_df['GPT-2'], width, label='GPT-2 (BPE)', color='#FF6B6B')
bars2 = ax.bar(x, comparison_df['mBERT'], width, label='mBERT (WordPiece)', color='#4ECDC4')
bars3 = ax.bar(x + width, comparison_df['XLM'], width, label='XLM (SentencePiece)', color='#45B7D1')

ax.set_xlabel('Text Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Token Count', fontsize=12, fontweight='bold')
ax.set_title('Token Count Comparison: BPE vs WordPiece vs SentencePiece', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Language'])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# 값 표시
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n그래프 표시 완료!")

---

## 8. 핵심 정리

### 8.1 관찰 결과

1. **영어**: BPE가 가장 효율적
2. **한글**: SentencePiece 또는 WordPiece가 더 효율적
3. **혼합**: SentencePiece가 최적

### 8.2 선택 기준

| 상황 | 추천 | 이유 |
|------|------|------|
| 영어만 | BPE | 효율성 |
| 한글 포함 | WordPiece 또는 SentencePiece | 한글 처리 최적화 |
| 다국어 | SentencePiece | 언어 독립적 처리 |
| 비용 최소화 | SentencePiece | 가장 적은 토큰 |

### 8.3 다음 단계

- README.md에서 이론적 배경 복습
- 실제 프로젝트에서 토크나이저 선택 실습
- 각 토크나이저의 특수 토큰과 설정 탐구